# 13 - The Conformal Model: Operational Euclidean Geometry

Source orientation: the PDF/page audit in this notebook records the printed and PDF page span used for chapter grounding; all prose, examples, visuals, and checks here are original course material.

This standalone notebook treats Chapter 13 as an executable lab for conformal geometric algebra in three Euclidean dimensions. The goal is not to reproduce the book's exposition. The goal is to make the operational idea tangible: lift a Euclidean point into a five-dimensional null vector space, let the metric carry distances, let simple vectors probe points, planes, and spheres, and let Euclidean motions act as metric-preserving linear transformations that keep the point at infinity fixed.

The chapter sits at a useful turn in the course. The homogeneous model already made finite points and points at infinity part of one linear story, but it did not make Euclidean distance a native inner-product measurement. The conformal model pays for two extra dimensions and an indefinite metric. In return, squared distance becomes an inner product of null point representatives, translations become part of the same orthogonal family as rotations, and reflection formulas become uniform enough to apply to points, flats, rounds, and later objects without rewriting the geometry for each case.

This notebook keeps the computations deliberately visible. We use the coordinate order `[e1, e2, e3, no, ni]`, where `no` is the origin-like null basis vector and `ni` is the point at infinity. A Euclidean point `x` is embedded as `X = x + no + 0.5 |x|^2 ni`. In this basis, the Euclidean part has the ordinary positive metric, while `no.ni = -1` and both `no` and `ni` square to zero. That small metric table is enough to explain why `X.X = 0` and why `-2 X.Y` recovers the squared distance between the two represented Euclidean points after weight normalization.

The notebook uses numerical helpers in `utils.chapter13_conformal`. Those helpers are not a full conformal algebra implementation. They are a practical bridge: the vector-level model, dual plane and sphere probes, conformal action matrices for rigid motions, Plucker-style line data for flats and directions, and screw interpolation through homogeneous transforms. The text says when a calculation is a coordinate shadow of a full versor sandwich. That distinction matters. The lab is faithful to the operational invariants without pretending that a 5-by-5 matrix is itself a multivector.

## PDF Page Audit

Before writing this notebook I checked the local PDF span requested for Chapter 13. In this scan, extractor pages 382-423 correspond to printed pages 355-396. The first extracted page opens the chapter title, and the last extracted page is still inside the programming example for interpolation of rigid body motions. The heading audit is saved under the chapter artifact folder and loaded in the first code cells below.

The verified heading sequence is useful because Chapter 13 has a very particular arc. It starts with the conformal model and its representational metric, then specializes point vectors as null vectors. It then explains that general vectors can be read as dual planes or spheres, which is one of the first surprises of CGA: a single vector can act as a geometric predicate. The middle of the chapter moves from objects to operations, showing why Euclidean transformations are represented by versors and why preserving the infinity vector is the defining restriction for Euclidean behavior inside the broader conformal group. The later sections revisit flats, directions, planar reflection, rigid body motions, screw motion, logarithms, interpolation, and differential reflection.

This notebook follows that arc with fresh prose and executable examples. It does not include the chapter's exercises verbatim. Instead, the final sanity checks encode the same habits the exercises are trying to build: verify nullness, recover distances from the inner product, test that operators preserve the metric and infinity, check reflection as an involution, and compare an analytic differential reflection formula against a finite-difference approximation.

In [1]:
from __future__ import annotations

from pathlib import Path
import json
import math
import sys

import nbformat
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import Markdown, display

BOOK_ROOT = Path.cwd()
for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "00-book-index.ipynb").exists() and (candidate / "utils" / "chapter13_conformal.py").exists():
        BOOK_ROOT = candidate
        break

if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

from utils.artifacts import display_artifact, save_json, save_plotly_html
from utils.chapter13_conformal import (
    METRIC,
    NI,
    NO,
    conformal_inner,
    conformal_norm2,
    conformal_point,
    conformal_weight,
    distance_squared_from_inner,
    dual_plane,
    dual_sphere,
    finite_difference_reflection_velocity,
    fractional_transform,
    infinity_error,
    line_from_points,
    metric_error,
    plane_probe,
    probe_table,
    recover_point,
    reflect_points,
    reflection_rt,
    reflection_velocity,
    rigid_cga_matrix,
    rotating_plane,
    rotation_matrix,
    sanity_checks,
    screw_parameters,
    screw_transform,
    sphere_probe,
    transform_line,
)

np.set_printoptions(precision=5, suppress=True)
ARTIFACT_ROOT = BOOK_ROOT / "artifacts" / "chapter-13"
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
NOTEBOOK_NAME = "13-conformal-model-operational-euclidean-geometry.ipynb"

heading_path = ARTIFACT_ROOT / "pdf-probe" / "heading-audit.json"
heading_audit = json.loads(heading_path.read_text(encoding="utf-8"))
print("project root:", BOOK_ROOT)
print("artifact root:", ARTIFACT_ROOT)
print("verified extractor pages:", heading_audit["verified_extractor_pages"])

project root: Geometric-Algebra-for-Computer-Science
artifact root: Geometric-Algebra-for-Computer-Science/artifacts/chapter-13
verified extractor pages: 382-423


In [2]:
heading_rows = pd.DataFrame(heading_audit["headings"])
display(heading_rows[["printed_page", "extractor_page", "heading"]])

summary = {
    "requested_printed_pages": heading_audit["requested_printed_pages"],
    "verified_extractor_pages": heading_audit["verified_extractor_pages"],
    "page_count": heading_audit["page_count"],
    "heading_count": len(heading_rows),
}
path = save_json(summary, "checks", None, "pdf-heading-summary.json", root=ARTIFACT_ROOT)
print(f"saved {path}")
summary

,printed_page,extractor_page,heading
0,355,382,13 THE CONFORMAL MODEL:
1,355,382,OPERATIONAL EUCLIDEAN
2,355,382,GEOMETRY
3,356,383,13.1 THE CONFORMAL MODEL
4,356,383,13.1.1 REPRESENTATIONAL SPACE AND METRIC
5,359,386,13.1.2 POINTS AS NULL VECTORS
6,361,388,13.1.3 GENERAL VECTORS REPRESENT DUAL PLANES
7,364,391,13.2 EUCLIDEAN TRANSFORMATIONS AS VERSORS
8,364,391,13.2.1 EUCLIDEAN VERSORS
9,365,392,13.2.2 PROPER EUCLIDEAN MOTIONS AS EVEN VERSORS


saved Geometric-Algebra-for-Computer-Science/artifacts/chapter-13/checks/pdf-heading-summary.json


{'requested_printed_pages': '355-396',
 'verified_extractor_pages': '382-423',
 'page_count': 42,
 'heading_count': 33}

## Metric And Null Embedding

The conformal model begins by changing the job of a vector. In an ordinary Euclidean vector-space model, a vector is primarily a displacement or direction. In the conformal model, a vector may instead be a point representative. That is possible because the metric is no longer positive definite. The two added basis directions are null-related: `no.no = 0`, `ni.ni = 0`, and `no.ni = -1`. A finite point is represented by a vector with a Euclidean part, a fixed `no` weight, and a quadratic `ni` coefficient.

The embedding has three immediate tests. First, every finite point representative must be null. Second, the weight of a finite point is recovered by `-X.ni`; for the normalized embedding used here that weight is one. Third, when two normalized point representatives are paired by the conformal inner product, the result is negative one half of the ordinary squared Euclidean distance. Those three facts are the small engine behind much of the chapter. If they fail, later formulas may still draw something, but they are no longer conformal geometry.

There is a geometric reason the formula looks quadratic. Euclidean squared distance between `x` and `y` is `|x-y|^2`. Expanding it gives `|x|^2 + |y|^2 - 2 x.y`. The conformal point stores `|x|^2` in the infinity coefficient and uses the null cross term between `no` and `ni` to make the expansion appear as a bilinear inner product. The model does not remove the quadratic term. It relocates it into a coordinate so that later computations can be linear in the representational space.

In [3]:
basis_names = ["e1", "e2", "e3", "no", "ni"]
metric_table = pd.DataFrame(METRIC, index=basis_names, columns=basis_names)
display(metric_table)

sample_points = np.array(
    [
        [-1.0, 0.25, 0.0],
        [0.0, 0.0, 0.0],
        [0.75, -0.5, 1.0],
        [1.25, 0.9, -0.4],
    ]
)
rows = []
for point in sample_points:
    X = conformal_point(point)
    rows.append(
        {
            "point": point.tolist(),
            "embedded": X.tolist(),
            "weight_-X_dot_ni": conformal_weight(X),
            "null_residual_X_dot_X": conformal_norm2(X),
            "recovered_point": recover_point(X).tolist(),
        }
    )

path = save_json(rows, "checks", None, "null-embedding.json", root=ARTIFACT_ROOT)
assert max(abs(row["null_residual_X_dot_X"]) for row in rows) < 1e-10
print(f"saved {path}")
display(pd.DataFrame(rows))

,e1,e2,e3,no,ni
e1,1.0,0.0,0.0,0.0,0.0
e2,0.0,1.0,0.0,0.0,0.0
e3,0.0,0.0,1.0,0.0,0.0
no,0.0,0.0,0.0,0.0,-1.0
ni,0.0,0.0,0.0,-1.0,0.0


saved Geometric-Algebra-for-Computer-Science/artifacts/chapter-13/checks/null-embedding.json


,point,embedded,weight_-X_dot_ni,null_residual_X_dot_X,recovered_point
0,"[-1.0, 0.25, 0.0]","[-1.0, 0.25, 0.0, 1.0, 0.53125]",1.0,0.0,"[-1.0, 0.25, 0.0]"
1,"[0.0, 0.0, 0.0]","[0.0, 0.0, 0.0, 1.0, 0.0]",1.0,0.0,"[0.0, 0.0, 0.0]"
2,"[0.75, -0.5, 1.0]","[0.75, -0.5, 1.0, 1.0, 0.90625]",1.0,0.0,"[0.75, -0.5, 1.0]"
3,"[1.25, 0.9, -0.4]","[1.25, 0.9, -0.4, 1.0, 1.26625]",1.0,0.0,"[1.25, 0.9, -0.4]"


In [4]:
grid = np.linspace(-2.0, 2.0, 45)
Xg, Yg = np.meshgrid(grid, grid)
Zg = 0.5 * (Xg**2 + Yg**2)

fig = go.Figure()
fig.add_trace(
    go.Surface(
        x=Xg,
        y=Yg,
        z=Zg,
        colorscale="Viridis",
        opacity=0.78,
        showscale=False,
        name="ni coefficient",
    )
)
fig.add_trace(
    go.Scatter3d(
        x=sample_points[:, 0],
        y=sample_points[:, 1],
        z=[0.5 * np.dot(p, p) for p in sample_points],
        mode="markers+text",
        marker=dict(size=5, color="#d62728"),
        text=[f"P{i}" for i in range(len(sample_points))],
        textposition="top center",
        name="embedded samples",
    )
)
fig.update_layout(
    title="Null point embedding shown as the stored infinity coefficient",
    scene=dict(
        xaxis_title="e1 coordinate",
        yaxis_title="e2 coordinate",
        zaxis_title="ni coefficient when z=0 and no=1",
        aspectmode="cube",
    ),
    margin=dict(l=0, r=0, t=45, b=0),
)
plot_path = save_plotly_html(fig, "plots", "null-embedding", "null-embedding.html", root=ARTIFACT_ROOT)
print(f"saved {plot_path}")
display_artifact(plot_path, height=520)

saved Geometric-Algebra-for-Computer-Science/artifacts/chapter-13/plots/null-embedding/null-embedding.html


## Distance Recovery

The distance identity is a good place to see why CGA is operational rather than merely representational. Once a point is lifted, we do not call a special distance function on the original Euclidean coordinates. We take an inner product in the representational space. The metric remembers how the Euclidean part and the null part should interact. That means any metric-preserving conformal operator will automatically preserve the recovered distances between transformed points.

The same identity also explains the role of weights. In projective and conformal settings, vectors often represent geometry up to scale. A point vector with weight two and the same Euclidean location is still the same finite point, but raw inner products scale with the two weights. The normalized identity divides out the weights before it interprets the value as distance. The helper function does that normalization explicitly, so the next table checks both ordinary points and scaled point vectors.

This habit is useful later when a motor or reflection acts on an object. If the result is scaled by the representation, the Euclidean meaning should be recovered after weight normalization. Treating weight as part of the model keeps numerical code honest: it prevents us from silently comparing a representative vector with the geometric object it names.

In [5]:
points = np.array(
    [
        [-1.2, 0.0, 0.4],
        [0.0, 1.0, -0.3],
        [0.8, -0.6, 1.2],
        [1.5, 0.75, 0.2],
    ]
)
D_cga = np.zeros((len(points), len(points)))
D_euclidean = np.zeros_like(D_cga)
for i, a in enumerate(points):
    for j, b in enumerate(points):
        scale_a = 1.0 + 0.2 * i
        scale_b = 1.0 + 0.1 * j
        A = scale_a * conformal_point(a)
        B = scale_b * conformal_point(b)
        D_cga[i, j] = distance_squared_from_inner(A, B)
        D_euclidean[i, j] = float(np.dot(a - b, a - b))

residual = np.abs(D_cga - D_euclidean)
path = save_json(
    {
        "cga_distance_squared": D_cga.tolist(),
        "euclidean_distance_squared": D_euclidean.tolist(),
        "max_abs_residual": float(residual.max()),
    },
    "checks",
    None,
    "distance-recovery.json",
    root=ARTIFACT_ROOT,
)
assert residual.max() < 1e-10
print(f"saved {path}")
display(pd.DataFrame(D_cga, columns=[f"P{j}" for j in range(len(points))], index=[f"P{i}" for i in range(len(points))]))

saved Geometric-Algebra-for-Computer-Science/artifacts/chapter-13/checks/distance-recovery.json


,P0,P1,P2,P3
P0,-0.0000,2.9300,5.0000,7.892500e+00
P1,2.9300,-0.0000,5.4500,2.562500e+00
P2,5.0000,5.4500,-0.0000,3.312500e+00
P3,7.8925,2.5625,3.3125,-8.881784e-16


## Vector Probes For Points, Planes, And Spheres

One of the first rewards of the conformal metric is that simple vectors can act as geometric probes. A normalized point vector `P` probes another point `X` by returning a value proportional to squared distance. A dual plane vector `pi = n + h ni`, with `n.x = h`, probes a point by signed distance. A dual sphere vector `S = C - 0.5 r^2 ni` probes a point by sphere power, the value `|x-c|^2-r^2`. Zero means incidence: the point lies on the plane or sphere. Positive and negative values distinguish sides or inside and outside.

This is the operational tone of the chapter. Instead of asking each object type for a different predicate, we encode the object as a vector and use the same inner product. Later chapters extend this idea to point pairs, circles, tangents, and intersections, but the vector probes are already enough to explain why the model feels compact. A single algebraic interaction can answer multiple geometric questions because the object's meaning is carried by the vector that represents it.

The next cells evaluate the three probes on a cloud of sample points. The saved Plotly figure shows a plane, a sphere, and a colored point cloud. Points near the plane have near-zero signed distance. Points near the sphere have near-zero power. The anchored point probe gives squared distance to a chosen sample point. Nothing in the code branches on a separate object hierarchy. The different geometric tests differ only in the vector used on the right side of the inner product.

In [6]:
anchor = np.array([-0.75, 0.4, 0.2])
plane = dual_plane([0.25, -0.65, 1.0], 0.2)
sphere_center = np.array([0.55, -0.25, 0.35])
sphere_radius = 0.95
sphere = dual_sphere(sphere_center, sphere_radius)

cloud = []
for x in np.linspace(-1.5, 1.5, 7):
    for y in np.linspace(-1.25, 1.25, 7):
        z = 0.35 * math.sin(1.7 * x) - 0.2 * math.cos(1.3 * y)
        cloud.append([x, y, z])
rows = probe_table(cloud, anchor=anchor, plane=plane, sphere=sphere)
probe_df = pd.DataFrame(rows)
path = save_json(rows, "checks", None, "vector-probes.json", root=ARTIFACT_ROOT)
print(f"saved {path}")
display(probe_df.head(10))
print("sphere power min/max:", probe_df["sphere_power"].min(), probe_df["sphere_power"].max())
print("plane signed distance min/max:", probe_df["plane_signed_distance"].min(), probe_df["plane_signed_distance"].max())

saved Geometric-Algebra-for-Computer-Science/artifacts/chapter-13/checks/vector-probes.json


,x,y,z,point_distance_squared,plane_signed_distance,sphere_power
0,-1.5,-1.250000,-0.184354,3.432728,0.043612,4.585534
1,-1.5,-0.833333,-0.288866,2.322602,-0.264400,4.048428
2,-1.5,-0.416667,-0.366559,1.550434,-0.550404,3.841235
3,-1.5,0.000000,-0.395189,1.076750,-0.796147,3.917807
4,-1.5,0.416667,-0.366559,0.883767,-0.994902,4.257902
5,-1.5,0.833333,-0.288866,0.989268,-1.153394,4.881761
6,-1.5,1.250000,-0.184354,1.432728,-1.289879,5.835534
7,-1.0,-1.250000,-0.336247,3.072561,0.021543,2.970935
8,-1.0,-0.833333,-0.440760,1.994184,-0.286469,2.465579
9,-1.0,-0.416667,-0.518453,1.245619,-0.572473,2.281988


sphere power min/max: -0.8054403036965261 5.8355340627460235
plane signed distance min/max: -1.3119478901368855 1.001487232398543


In [7]:
cloud_array = np.asarray(cloud)
n = plane[:3]
h = plane[4]
plane_x, plane_y = np.meshgrid(np.linspace(-1.6, 1.6, 12), np.linspace(-1.4, 1.4, 12))
plane_z = (h - n[0] * plane_x - n[1] * plane_y) / n[2]

u = np.linspace(0, 2 * np.pi, 32)
v = np.linspace(0, np.pi, 16)
sx = sphere_center[0] + sphere_radius * np.outer(np.cos(u), np.sin(v))
sy = sphere_center[1] + sphere_radius * np.outer(np.sin(u), np.sin(v))
sz = sphere_center[2] + sphere_radius * np.outer(np.ones_like(u), np.cos(v))

fig = go.Figure()
fig.add_trace(
    go.Scatter3d(
        x=cloud_array[:, 0],
        y=cloud_array[:, 1],
        z=cloud_array[:, 2],
        mode="markers",
        marker=dict(size=5, color=probe_df["sphere_power"], colorscale="RdBu", colorbar=dict(title="sphere power")),
        name="probe samples",
    )
)
fig.add_trace(go.Surface(x=plane_x, y=plane_y, z=plane_z, opacity=0.32, showscale=False, colorscale="Greens", name="dual plane"))
fig.add_trace(go.Surface(x=sx, y=sy, z=sz, opacity=0.24, showscale=False, colorscale="Blues", name="dual sphere"))
fig.add_trace(go.Scatter3d(x=[anchor[0]], y=[anchor[1]], z=[anchor[2]], mode="markers+text", marker=dict(size=7, color="black"), text=["point probe"], name="anchor"))
fig.update_layout(
    title="Point, plane, and sphere probes evaluated by conformal inner products",
    scene=dict(xaxis_title="x", yaxis_title="y", zaxis_title="z", aspectmode="cube"),
    margin=dict(l=0, r=0, t=45, b=0),
)
plot_path = save_plotly_html(fig, "plots", "vector-probes", "vector-probes.html", root=ARTIFACT_ROOT)
print(f"saved {plot_path}")
display_artifact(plot_path, height=540)

saved Geometric-Algebra-for-Computer-Science/artifacts/chapter-13/plots/vector-probes/vector-probes.html


## Versors That Preserve Infinity

The full conformal group is larger than the Euclidean group. Chapter 13 focuses on the Euclidean part first. The coordinate test for that restriction is simple: a Euclidean versor preserves the point at infinity. In the full algebra this is a sandwich action. In this notebook we use its 5-by-5 coordinate shadow for a rigid map `x -> R x + t`. The matrix is built so that embedded points go to embedded points, the conformal metric is preserved, and `ni` is fixed.

Preserving `ni` is more than a bookkeeping condition. It means the transformation respects the Euclidean structure hidden inside the conformal model. If an operator moved the infinity vector, it could still be conformal, but it would no longer be an ordinary Euclidean motion. It might become an inversion, dilation, or other conformal transformation studied later. In Chapter 13, preserving infinity is the gate that keeps rotations, translations, reflections, and motors in the Euclidean family.

The next check uses a rotation and translation. Distances between a small set of points are computed before and after the action. The conformal matrix is tested against the metric matrix, and its action on `ni` is checked directly. The important lesson is that distance preservation is not verified by a pile of individual coordinate formulas. Once the operator preserves the conformal metric and infinity, the distance identity from the previous section makes distance preservation a consequence.

In [8]:
axis = np.array([0.2, 0.45, 1.0])
R = rotation_matrix(axis, 0.85)
t = np.array([0.65, -0.35, 0.45])
M = rigid_cga_matrix(R, t)

moved_points = (R @ points.T).T + t
before = np.array([[float(np.dot(a - b, a - b)) for b in points] for a in points])
after = np.array([[float(np.dot(a - b, a - b)) for b in moved_points] for a in moved_points])
point_action_residuals = []
for point in points:
    moved_cga = M @ conformal_point(point)
    moved_expected = conformal_point(R @ point + t)
    point_action_residuals.append(float(np.linalg.norm(moved_cga - moved_expected)))

versor_checks = {
    "metric_error": metric_error(M),
    "infinity_error": infinity_error(M),
    "max_point_action_residual": max(point_action_residuals),
    "max_distance_residual": float(np.abs(before - after).max()),
    "determinant_rotation": float(np.linalg.det(R)),
}
path = save_json(versor_checks, "checks", None, "versor-preserves-infinity.json", root=ARTIFACT_ROOT)
assert versor_checks["metric_error"] < 1e-10
assert versor_checks["infinity_error"] < 1e-10
assert versor_checks["max_distance_residual"] < 1e-10
print(f"saved {path}")
versor_checks

saved Geometric-Algebra-for-Computer-Science/artifacts/chapter-13/checks/versor-preserves-infinity.json


{'metric_error': 3.9340370205249993e-16,
 'infinity_error': 0.0,
 'max_point_action_residual': 2.2887833992611187e-16,
 'max_distance_residual': 2.6645352591003757e-15,
 'determinant_rotation': 1.0000000000000002}

## Flats And Directions

Flats are the conformal continuation of the homogeneous model's lines, planes, and directions. The full direct representation uses outer products of embedded points and the infinity vector. This notebook stays at the coordinate level and records the same information for a 3-D line as a unit direction plus a Plucker moment. That pair is not a conformal blade, but it is the Euclidean data carried by one. It makes the chapter's direction claim easy to see: translating a line changes its moment, because the line's offset from the origin changes, but it does not change its direction.

The point at infinity is the shared ingredient that lets directions become location-free. A finite point changes under translation. A line's nearest point to the origin changes under translation. But the direction component belongs to the line's ideal part, and a Euclidean translation preserves that ideal component. Rotations act on directions, translations do not. This is exactly the behavior expected from preserving `ni`.

In a full CGA implementation, a direct line through two points would be something like `P(a) ^ P(b) ^ ni`, and a plane through three points would include `ni` as well. A direction can be inspected by looking at the infinity-containing part of the blade. The code below shows the same operational behavior with Plucker data and a saved figure. It draws a line, translates it, then rotates and translates it. The direction comparison separates what is intrinsic to the flat from what is caused by choosing an origin.

In [9]:
line = line_from_points([-1.2, -0.4, 0.1], [0.8, 0.9, 0.7])
translation_only = np.array([0.9, -0.2, 0.45])
line_translated = transform_line(line, np.eye(3), translation_only)
line_moved = transform_line(line, R, t)

def line_segment(line_obj, span=1.8, count=25):
    q = line_obj.point_nearest_origin()
    s = np.linspace(-span, span, count)
    return q[None, :] + s[:, None] * line_obj.direction[None, :]

line_checks = {
    "original_direction": line.direction.tolist(),
    "translated_direction": line_translated.direction.tolist(),
    "moved_direction": line_moved.direction.tolist(),
    "translation_direction_error": float(np.linalg.norm(line.direction - line_translated.direction)),
    "rotation_direction_error": float(np.linalg.norm(R @ line.direction - line_moved.direction)),
    "original_moment": line.moment.tolist(),
    "translated_moment": line_translated.moment.tolist(),
}
path = save_json(line_checks, "checks", None, "flats-directions.json", root=ARTIFACT_ROOT)
assert line_checks["translation_direction_error"] < 1e-12
assert line_checks["rotation_direction_error"] < 1e-12
print(f"saved {path}")
line_checks

saved Geometric-Algebra-for-Computer-Science/artifacts/chapter-13/checks/flats-directions.json


{'original_direction': [0.8131156281817417,
  0.5285251583181321,
  0.24393468845452249],
 'translated_direction': [0.8131156281817417,
  0.5285251583181321,
  0.24393468845452249],
 'moved_direction': [0.2896743546049212,
  0.9433219524724915,
  0.16196438580042485],
 'translation_direction_error': 0.0,
 'rotation_direction_error': 0.0,
 'original_moment': [-0.15042639121362222,
  0.37403318896360116,
  -0.3089839387090618],
 'translated_moment': [-0.43704965014768615,
  0.5203940020363147,
  0.3293118294136054]}

In [10]:
seg0 = line_segment(line)
seg1 = line_segment(line_translated)
seg2 = line_segment(line_moved)
fig = go.Figure()
for seg, name, color in [(seg0, "original line", "#1f77b4"), (seg1, "translated line", "#ff7f0e"), (seg2, "rotated and translated line", "#2ca02c")]:
    fig.add_trace(go.Scatter3d(x=seg[:, 0], y=seg[:, 1], z=seg[:, 2], mode="lines", line=dict(width=6, color=color), name=name))
fig.add_trace(go.Cone(x=[seg0[12, 0], seg1[12, 0], seg2[12, 0]], y=[seg0[12, 1], seg1[12, 1], seg2[12, 1]], z=[seg0[12, 2], seg1[12, 2], seg2[12, 2]], u=[line.direction[0], line_translated.direction[0], line_moved.direction[0]], v=[line.direction[1], line_translated.direction[1], line_moved.direction[1]], w=[line.direction[2], line_translated.direction[2], line_moved.direction[2]], sizemode="absolute", sizeref=0.35, showscale=False, name="directions"))
fig.update_layout(
    title="Flats carry both offset data and an ideal direction component",
    scene=dict(xaxis_title="x", yaxis_title="y", zaxis_title="z", aspectmode="cube"),
    margin=dict(l=0, r=0, t=45, b=0),
)
plot_path = save_plotly_html(fig, "plots", "flats-directions", "flats-directions.html", root=ARTIFACT_ROOT)
print(f"saved {plot_path}")
display_artifact(plot_path, height=520)

saved Geometric-Algebra-for-Computer-Science/artifacts/chapter-13/plots/flats-directions/flats-directions.html


## General Planar Reflection

A plane reflection is the simplest example where the conformal model starts to feel like a geometry engine. In ordinary vector algebra, reflecting a point in the plane `n.x = h` is already easy: subtract twice the signed distance along the normal. In CGA, the dual plane vector is also a reflecting vector, and the same sandwich pattern applies to any represented element, not just a finite point. This is why the chapter treats planar reflection as an application rather than as a detached formula.

The coordinate version below reflects a cube of sample points. The checks are intentionally redundant. Signed plane distances should change sign. Applying the reflection twice should return the original points. The conformal action matrix for the reflection should preserve the conformal metric and keep `ni` fixed. Its determinant in Euclidean space is negative, so it is not a proper motion, but it is still a Euclidean isometry. In geometric-algebra language it belongs with the versors even though it is not an even motor.

This distinction is worth keeping. Motors are built from even products and describe rigid body motions that can be performed continuously from the identity. A single reflection reverses handedness. It is still metric preserving, but it cannot be reached by gradually rotating and translating an object without leaving the proper motion group. CGA makes both families operational, while preserving the difference in their algebraic parity.

In [11]:
cube = np.array([[x, y, z] for x in [-0.5, 0.5] for y in [-0.5, 0.5] for z in [-0.5, 0.5]], dtype=float)
plane_normal = np.array([0.45, -0.8, 0.4])
plane_offset = 0.15
R_ref, t_ref = reflection_rt(plane_normal, plane_offset)
reflected_cube = reflect_points(cube, plane_normal, plane_offset)
M_ref = rigid_cga_matrix(R_ref, t_ref)
pi_ref = dual_plane(plane_normal, plane_offset)
original_signed = np.array([plane_probe(p, pi_ref) for p in cube])
reflected_signed = np.array([plane_probe(p, pi_ref) for p in reflected_cube])
round_trip = reflect_points(reflected_cube, plane_normal, plane_offset)
reflection_checks = {
    "metric_error": metric_error(M_ref),
    "infinity_error": infinity_error(M_ref),
    "determinant": float(np.linalg.det(R_ref)),
    "signed_distance_flip_error": float(np.max(np.abs(original_signed + reflected_signed))),
    "round_trip_error": float(np.linalg.norm(round_trip - cube)),
}
path = save_json(reflection_checks, "checks", None, "planar-reflection.json", root=ARTIFACT_ROOT)
assert reflection_checks["metric_error"] < 1e-10
assert reflection_checks["signed_distance_flip_error"] < 1e-10
assert reflection_checks["round_trip_error"] < 1e-10
print(f"saved {path}")
reflection_checks

saved Geometric-Algebra-for-Computer-Science/artifacts/chapter-13/checks/planar-reflection.json


{'metric_error': 1.3087147758658702e-16,
 'infinity_error': 0.0,
 'determinant': -1.0,
 'signed_distance_flip_error': 2.220446049250313e-16,
 'round_trip_error': 2.482534153247273e-16}

In [12]:
n = pi_ref[:3]
h = pi_ref[4]
plane_x, plane_y = np.meshgrid(np.linspace(-1.2, 1.4, 8), np.linspace(-1.2, 1.2, 8))
plane_z = (h - n[0] * plane_x - n[1] * plane_y) / n[2]
fig = go.Figure()
fig.add_trace(go.Scatter3d(x=cube[:, 0], y=cube[:, 1], z=cube[:, 2], mode="markers", marker=dict(size=6, color="#1f77b4"), name="original cube points"))
fig.add_trace(go.Scatter3d(x=reflected_cube[:, 0], y=reflected_cube[:, 1], z=reflected_cube[:, 2], mode="markers", marker=dict(size=6, color="#d62728"), name="reflected cube points"))
for a, b in zip(cube, reflected_cube):
    pair = np.vstack([a, b])
    fig.add_trace(go.Scatter3d(x=pair[:, 0], y=pair[:, 1], z=pair[:, 2], mode="lines", line=dict(width=2, color="rgba(80,80,80,0.35)"), showlegend=False))
fig.add_trace(go.Surface(x=plane_x, y=plane_y, z=plane_z, opacity=0.35, showscale=False, colorscale="Greys", name="reflection plane"))
fig.update_layout(
    title="Planar reflection as a conformal metric-preserving action",
    scene=dict(xaxis_title="x", yaxis_title="y", zaxis_title="z", aspectmode="cube"),
    margin=dict(l=0, r=0, t=45, b=0),
)
plot_path = save_plotly_html(fig, "plots", "planar-reflection", "planar-reflection.html", root=ARTIFACT_ROOT)
print(f"saved {plot_path}")
display_artifact(plot_path, height=540)

saved Geometric-Algebra-for-Computer-Science/artifacts/chapter-13/plots/planar-reflection/planar-reflection.html

## Motors, Screws, And Interpolation

Proper Euclidean motions are compositions of rotations and translations. In conformal geometric algebra they are represented by even versors called motors. The chapter's key operational message is that motors can be handled like rotors in a larger space: compose them by multiplication, apply them by sandwiching, and interpolate them by taking logarithms and exponentials. The motion may look like a rotation plus a translation in coordinates, but Chasles' theorem says it can be viewed as a screw motion about an axis with a possible translation along that axis.

The code below constructs a screw from an axis, a point on the axis, an angle, and a pitch. It then recovers Chasles-style parameters from the resulting rigid transform. For interpolation, we use the matrix exponential and logarithm of the homogeneous 4-by-4 transform. This is not a replacement for a native motor logarithm, but it is the same Lie-group idea in a familiar coordinate representation. The endpoints are checked, and the saved Plotly figure shows a small frame moving along the screw path.

Why interpolate this way instead of linearly blending matrices or points? Linear blending does not generally stay in the group of rigid motions. It can shrink, shear, or drift away from orthogonality. A logarithm-based path follows the generator of the motion. In motor language, it gradually applies the bivector that generates the screw. In matrix language, it exponentiates a twist. Both readings preserve the idea that the halfway pose should still be a valid rigid pose.

In [13]:
screw_axis = np.array([0.15, 0.35, 1.0])
screw_axis = screw_axis / np.linalg.norm(screw_axis)
axis_point = np.array([0.35, -0.45, 0.1])
screw_angle = 1.25
screw_pitch = 0.28
R_screw, t_screw = screw_transform(screw_axis, axis_point, screw_angle, screw_pitch)
params = screw_parameters(R_screw, t_screw)
fractions = np.linspace(0.0, 1.0, 9)
probe_point = np.array([1.1, -0.2, 0.4])
path_points = []
endpoint_errors = []
for s in fractions:
    R_part, t_part = fractional_transform(R_screw, t_screw, float(s))
    path_points.append(R_part @ probe_point + t_part)
    if s in {0.0, 1.0}:
        expected = probe_point if s == 0.0 else R_screw @ probe_point + t_screw
        endpoint_errors.append(float(np.linalg.norm(path_points[-1] - expected)))

serial_params = {
    "kind": params["kind"],
    "axis": np.asarray(params["axis"]).tolist(),
    "angle": float(params["angle"]),
    "axis_point": np.asarray(params["axis_point"]).tolist(),
    "parallel_displacement": float(params["parallel_displacement"]),
    "pitch": None if not np.isfinite(params["pitch"]) else float(params["pitch"]),
    "max_endpoint_error": max(endpoint_errors),
}
path = save_json(serial_params, "checks", None, "screw-parameters.json", root=ARTIFACT_ROOT)
assert serial_params["max_endpoint_error"] < 1e-9
print(f"saved {path}")
serial_params

saved Geometric-Algebra-for-Computer-Science/artifacts/chapter-13/checks/screw-parameters.json


{'kind': 'screw',
 'axis': [0.1401807940547993, 0.32708851946119843, 0.9345386270319954],
 'angle': 1.25,
 'axis_point': [0.35065502183406133,
  -0.44847161572052374,
  0.10436681222707422],
 'parallel_displacement': 0.35,
 'pitch': 0.27999999999999997,
 'max_endpoint_error': 6.866350197783356e-16}

In [14]:
path_points = np.asarray(path_points)
axis_line_s = np.linspace(-1.6, 1.8, 20)
axis_line = axis_point[None, :] + axis_line_s[:, None] * screw_axis[None, :]
frame_origins = []
frame_axes = []
for s in np.linspace(0.0, 1.0, 6):
    R_part, t_part = fractional_transform(R_screw, t_screw, float(s))
    origin = R_part @ np.zeros(3) + t_part
    frame_origins.append(origin)
    frame_axes.append(R_part)
frame_origins = np.asarray(frame_origins)

fig = go.Figure()
fig.add_trace(go.Scatter3d(x=axis_line[:, 0], y=axis_line[:, 1], z=axis_line[:, 2], mode="lines", line=dict(width=5, color="black"), name="screw axis"))
fig.add_trace(go.Scatter3d(x=path_points[:, 0], y=path_points[:, 1], z=path_points[:, 2], mode="markers+lines", marker=dict(size=5, color=fractions, colorscale="Viridis", colorbar=dict(title="fraction")), line=dict(width=4, color="#9467bd"), name="interpolated point"))
axis_colors = ["#d62728", "#2ca02c", "#1f77b4"]
axis_names = ["local x", "local y", "local z"]
for frame_index, (origin, axes) in enumerate(zip(frame_origins, frame_axes)):
    for idx, color in enumerate(axis_colors):
        tip = origin + 0.28 * axes[:, idx]
        seg = np.vstack([origin, tip])
        fig.add_trace(go.Scatter3d(x=seg[:, 0], y=seg[:, 1], z=seg[:, 2], mode="lines", line=dict(width=4, color=color), name=axis_names[idx], showlegend=frame_index == 0))
fig.update_layout(
    title="Screw interpolation through fractional rigid transforms",
    scene=dict(xaxis_title="x", yaxis_title="y", zaxis_title="z", aspectmode="cube"),
    margin=dict(l=0, r=0, t=45, b=0),
)
plot_path = save_plotly_html(fig, "plots", "screw-interpolation", "screw-interpolation.html", root=ARTIFACT_ROOT)
print(f"saved {plot_path}")
display_artifact(plot_path, height=560)

saved Geometric-Algebra-for-Computer-Science/artifacts/chapter-13/plots/screw-interpolation/screw-interpolation.html


## Differential Planar Reflection

The chapter returns to differentiation by letting a reflecting plane move. A stationary plane reflection is an isometry. A rotating plane reflection is a one-parameter family of isometries, so it has a velocity field. The conformal view makes the setup natural because the mirror is a dual plane vector, and changing the mirror means differentiating the vector that participates in the reflection sandwich.

The coordinate version here rotates a plane around a fixed hinge line. The plane normal changes by `axis cross normal`. The plane offset also changes unless the hinge goes through the origin, because the same physical hinge point must remain in the plane. That offset derivative is exactly the detail that a pure vector-space mirror calculation tends to hide. Once the plane is not forced through the origin, the derivative needs both the changing normal and the changing offset.

The analytic formula differentiates `x' = x - 2 (n.x - h) n`. The derivative has two terms: one for the changing signed distance and one for the changing normal direction. The finite-difference check evaluates the reflected point slightly before and after the selected angle. Their difference should agree with the analytic velocity. The saved Plotly figure uses cones to show the reflected-point velocity field for a grid of sample points.

In [15]:
hinge_axis = np.array([0.0, 0.0, 1.0])
base_normal = np.array([1.0, 0.0, 0.0])
hinge_point = np.array([0.25, -0.35, 0.0])
mirror_angle = 0.65
mirror_points = np.array([[x, y, 0.2 * math.sin(2.0 * x)] for x in np.linspace(-1.0, 1.0, 6) for y in np.linspace(-1.0, 1.0, 6)])
normal_now, offset_now = rotating_plane(hinge_axis, base_normal, hinge_point, mirror_angle)
reflected_points = reflect_points(mirror_points, normal_now, offset_now)
analytic_velocity = reflection_velocity(mirror_points, hinge_axis, base_normal, hinge_point, mirror_angle)
numeric_velocity = finite_difference_reflection_velocity(mirror_points, hinge_axis, base_normal, hinge_point, mirror_angle)
velocity_error = float(np.linalg.norm(analytic_velocity - numeric_velocity))
velocity_checks = {
    "normal": normal_now.tolist(),
    "offset": float(offset_now),
    "velocity_error_norm": velocity_error,
    "max_velocity_component": float(np.max(np.abs(analytic_velocity))),
}
path = save_json(velocity_checks, "checks", None, "differential-planar-reflection.json", root=ARTIFACT_ROOT)
assert velocity_error < 1e-8
print(f"saved {path}")
velocity_checks

saved Geometric-Algebra-for-Computer-Science/artifacts/chapter-13/checks/differential-planar-reflection.json


{'normal': [0.7960837985490559, 0.6051864057360395, 0.0],
 'offset': -0.012794292370349875,
 'velocity_error_norm': 9.174167874649378e-10,
 'max_velocity_component': 3.1311423008293686}

In [16]:
px, py = np.meshgrid(np.linspace(-1.2, 1.2, 8), np.linspace(-1.2, 1.2, 8))
if abs(normal_now[2]) > 1e-8:
    pz = (offset_now - normal_now[0] * px - normal_now[1] * py) / normal_now[2]
else:
    pz = np.zeros_like(px)
fig = go.Figure()
fig.add_trace(go.Surface(x=px, y=py, z=pz, opacity=0.28, colorscale="Greys", showscale=False, name="moving mirror"))
fig.add_trace(go.Scatter3d(x=reflected_points[:, 0], y=reflected_points[:, 1], z=reflected_points[:, 2], mode="markers", marker=dict(size=4, color="#1f77b4"), name="reflected points"))
fig.add_trace(
    go.Cone(
        x=reflected_points[:, 0],
        y=reflected_points[:, 1],
        z=reflected_points[:, 2],
        u=analytic_velocity[:, 0],
        v=analytic_velocity[:, 1],
        w=analytic_velocity[:, 2],
        sizemode="absolute",
        sizeref=0.35,
        colorscale="Inferno",
        showscale=False,
        name="reflection velocity",
    )
)
fig.update_layout(
    title="Differential planar reflection from a rotating offset mirror",
    scene=dict(xaxis_title="x", yaxis_title="y", zaxis_title="z", aspectmode="cube"),
    margin=dict(l=0, r=0, t=45, b=0),
)
plot_path = save_plotly_html(fig, "plots", "differential-planar-reflection", "differential-planar-reflection.html", root=ARTIFACT_ROOT)
print(f"saved {plot_path}")
display_artifact(plot_path, height=560)

saved Geometric-Algebra-for-Computer-Science/artifacts/chapter-13/plots/differential-planar-reflection/differential-planar-reflection.html


## Final Sanity Checks

A conformal notebook can look convincing while still being wrong in the metric. The final cell treats the notebook like a small numerical contract. It checks null embedding, distance recovery, metric preservation, infinity preservation, planar reflection, screw endpoints, differential reflection, and the authoring quality gate requested for this chapter. The artifact count is included because the figures are part of the notebook's teaching surface, not decoration.

The most important residual is the distance identity. If `-2 X.Y` does not match ordinary squared distance, then the chosen null basis or scaling is wrong. The next most important residual is infinity preservation for Euclidean motions. A matrix can preserve the conformal metric and still move `ni`; that would leave the Euclidean subgroup. Reflection gets an involution check because it should undo itself. Differential reflection gets a finite-difference check because derivative formulas are easy to misread when the plane is offset from the origin.

The notebook is intentionally seed-style: it is a single, standalone lab that can be extended into more formal exercises later. The helper module contains only the reusable numerical pieces. The prose and code cells here carry the chapter narrative from the metric through vectors, flats, reflections, motors, interpolation, and derivatives.

In [17]:
checks = sanity_checks()
plotly_artifacts = sorted(str(path.relative_to(BOOK_ROOT)) for path in (ARTIFACT_ROOT / "plots").rglob("*.html"))
nb_path = Path(NOTEBOOK_NAME)
nb = nbformat.read(nb_path, as_version=4)
markdown_words = sum(len(cell.source.split()) for cell in nb.cells if cell.cell_type == "markdown")
code_cells = sum(1 for cell in nb.cells if cell.cell_type == "code")
quality = {
    "markdown_words": markdown_words,
    "code_cells": code_cells,
    "plotly_artifacts": plotly_artifacts,
    "plotly_artifact_count": len(plotly_artifacts),
    "numeric_checks": checks,
}
path = save_json(quality, "checks", None, "final-sanity-checks.json", root=ARTIFACT_ROOT)

thresholds = {
    "markdown_words": markdown_words >= 2500,
    "code_cells": code_cells >= 10,
    "plotly_artifact_count": len(plotly_artifacts) >= 5,
    "point_null_residual": checks["point_null_residual"] < 1e-10,
    "distance_identity_residual": checks["distance_identity_residual"] < 1e-10,
    "metric_preservation_error": checks["metric_preservation_error"] < 1e-10,
    "infinity_preservation_error": checks["infinity_preservation_error"] < 1e-10,
    "reflection_involution_error": checks["reflection_involution_error"] < 1e-10,
    "differential_reflection_error": checks["differential_reflection_error"] < 1e-7,
}
failed = [name for name, ok in thresholds.items() if not ok]
if failed:
    raise AssertionError(f"quality gate failed: {failed}")

print(f"saved {path}")
display(Markdown("\n".join(["| Check | Value |", "|---|---|"] + [f"| {k} | {v} |" for k, v in quality.items() if k != "plotly_artifacts"])))
print("artifacts:")
for artifact in plotly_artifacts:
    print(" -", artifact)

saved Geometric-Algebra-for-Computer-Science/artifacts/chapter-13/checks/final-sanity-checks.json


| Check | Value |
|---|---|
| markdown_words | 2755 |
| code_cells | 17 |
| plotly_artifact_count | 6 |
| numeric_checks | {'point_null_residual': 0.0, 'distance_identity_residual': 0.0, 'metric_preservation_error': 2.9486149327106574e-17, 'infinity_preservation_error': 0.0, 'plane_probe_after_reflection': 0.0, 'reflection_involution_error': 1.1102230246251565e-16, 'differential_reflection_error': 1.2229153039415618e-10} |

artifacts:
 - artifacts/chapter-13/plots/differential-planar-reflection/differential-planar-reflection.html
 - artifacts/chapter-13/plots/flats-directions/flats-directions.html
 - artifacts/chapter-13/plots/null-embedding/null-embedding.html
 - artifacts/chapter-13/plots/planar-reflection/planar-reflection.html
 - artifacts/chapter-13/plots/screw-interpolation/screw-interpolation.html
 - artifacts/chapter-13/plots/vector-probes/vector-probes.html


## Takeaways

- `13 - The Conformal Model: Operational Euclidean Geometry` is treated as an executable geometric algebra lesson: definitions are paired with concrete representations, artifacts, and checks.
- The chapter's computations make the model choices visible, so algebraic operations can be inspected instead of accepted as black-box notation.
- The saved artifacts and sanity checks are the audit trail for this standalone notebook; they support the text without reproducing textbook prose or figures.